# Polonia en Datos
Notebook de análisis — consume JSONs generados por los scripts de extracción.

## 0 · Config & Rutas

In [1]:
import json
import os
import glob
import pandas as pd

# Raíz del proyecto = directorio donde vive este notebook
ROOT = os.path.dirname(os.path.abspath('__file__'))  # Jupyter: ajustar si hace falta

PATHS = {
    'nbp':       os.path.join(ROOT, 'nbp', 'nbp_data'),
    'eurostat':  os.path.join(ROOT, 'eurostat', 'eurostat_data'),
    'worldbank': os.path.join(ROOT, 'worldbank', 'worldbank_data'),
}

# Verificar disponibilidad
for source, path in PATHS.items():
    files = glob.glob(os.path.join(path, '*.json'))
    print(f"{source:<12} → {path}")
    print(f"             {len(files)} archivo(s): {[os.path.basename(f) for f in files]}")

nbp          → /Users/raulrivera/Desktop/1-Data/Data Analyst Projects/poland_data2/nbp/nbp_data
             2 archivo(s): ['EUR_A_last_10.json', 'gold_last_5.json']
eurostat     → /Users/raulrivera/Desktop/1-Data/Data Analyst Projects/poland_data2/eurostat/eurostat_data
             2 archivo(s): ['eurostat_datos.json', 'eurostat_structure_cache.json']
worldbank    → /Users/raulrivera/Desktop/1-Data/Data Analyst Projects/poland_data2/worldbank/worldbank_data
             1 archivo(s): ['NY_GDP_MKTP_CD_PL_2014_2023.json']


## 1 · Loaders & Parsers

### 1.1 NBP

In [2]:
def load_nbp_rates(filename):
    """
    Parsea un archivo de tipos de cambio NBP.
    Retorna DataFrame: date | currency | table | rate
    """
    filepath = os.path.join(PATHS['nbp'], filename)
    with open(filepath, encoding='utf-8') as f:
        raw = json.load(f)
    
    df = pd.DataFrame(raw['rates'])
    df = df.rename(columns={'effectiveDate': 'date', 'mid': 'rate'})
    df['date'] = pd.to_datetime(df['date'])
    df['currency'] = raw['code']
    df['table'] = raw['table']
    return df[['date', 'currency', 'table', 'rate']]


def load_nbp_gold(filename):
    """
    Parsea un archivo de precios de oro NBP.
    Retorna DataFrame: date | gold_pln
    """
    filepath = os.path.join(PATHS['nbp'], filename)
    with open(filepath, encoding='utf-8') as f:
        raw = json.load(f)
    
    df = pd.DataFrame(raw)
    df = df.rename(columns={'data': 'date', 'cena': 'gold_pln'})
    df['date'] = pd.to_datetime(df['date'])
    return df[['date', 'gold_pln']]


# --- Cargar archivos disponibles ---
nbp_eur = load_nbp_rates('EUR_A_last_10.json')
nbp_gold = load_nbp_gold('gold_last_5.json')

print("NBP EUR:", nbp_eur.shape)
display(nbp_eur.head())

print("\nNBP Gold:", nbp_gold.shape)
display(nbp_gold.head())

NBP EUR: (10, 4)


,date,currency,table,rate
0,2026-06-10,EUR,A,4.2481
1,2026-06-11,EUR,A,4.2530
2,2026-06-12,EUR,A,4.2484
3,2026-06-15,EUR,A,4.2447
4,2026-06-16,EUR,A,4.2462



NBP Gold: (5, 2)


,date,gold_pln
0,2026-06-17,511.22
1,2026-06-18,507.97
2,2026-06-19,508.44
3,2026-06-22,498.01
4,2026-06-23,503.01


### 1.2 Eurostat

In [4]:
def load_eurostat(filename):
    """
    Parsea JSON-stat de Eurostat.
    Retorna DataFrame con todas las dimensiones como columnas + 'value'.
    """
    filepath = os.path.join(PATHS['eurostat'], filename)
    with open(filepath, encoding='utf-8') as f:
        raw = json.load(f)

    dims = raw['dimension']
    values = raw['value']  # {str_index: value}

    # Construir mapas: posición → código, código → label
    dim_names = list(dims.keys())
    dim_maps = []  # lista de {pos: code}
    dim_labels = []  # lista de {code: label}

    sizes = []
    for dim in dim_names:
        index = dims[dim]['category']['index']   # {code: pos}
        label = dims[dim]['category']['label']   # {code: label}
        pos_to_code = {v: k for k, v in index.items()}
        dim_maps.append(pos_to_code)
        dim_labels.append(label)
        sizes.append(len(index))

    # Reconstruir filas
    rows = []
    for str_idx, val in values.items():
        flat_idx = int(str_idx)
        row = {}
        for i in range(len(sizes) - 1, -1, -1):
            pos = flat_idx % sizes[i]
            flat_idx //= sizes[i]
            code = dim_maps[i].get(pos, pos)
            row[dim_names[i]] = code
            row[f"{dim_names[i]}_label"] = dim_labels[i].get(code, code)
        row['value'] = val
        rows.append(row)

    return pd.DataFrame(rows)


# --- Cargar ---
df_eurostat = load_eurostat('eurostat_datos.json')

print("Eurostat shape:", df_eurostat.shape)
print("Columnas:", df_eurostat.columns.tolist())
display(df_eurostat.head())

Eurostat shape: (717, 11)
Columnas: ['time', 'time_label', 'geo', 'geo_label', 'coicop', 'coicop_label', 'unit', 'unit_label', 'freq', 'freq_label', 'value']


,time,time_label,geo,geo_label,coicop,coicop_label,unit,unit_label,freq,freq_label,value
0,2023,2023,ES,Spain,TOT_X_NRG_FOOD,"Overall index excluding energy, food, alcohol ...",CID_EA,Core inflation differential vis-à-vis the euro...,A,Annual,-0.80
1,2023,2023,ES,Spain,AP,Administered prices,INX_A_AVG,Annual average index,A,Annual,101.74
2,2023,2023,ES,Spain,APF,Fully administered prices,INX_A_AVG,Annual average index,A,Annual,100.60
3,2023,2023,ES,Spain,APM,Mainly administered prices,INX_A_AVG,Annual average index,A,Annual,114.23
4,2023,2023,ES,Spain,AP_NNRG,"Administered prices, non-energy",INX_A_AVG,Annual average index,A,Annual,99.52


### 1.3 World Bank

In [5]:
def load_worldbank(filename):
    """
    Parsea JSON de World Bank (lista de records).
    Retorna DataFrame: country | country_code | indicator | indicator_code | year | value
    """
    filepath = os.path.join(PATHS['worldbank'], filename)
    with open(filepath, encoding='utf-8') as f:
        records = json.load(f)

    rows = []
    for r in records:
        rows.append({
            'country':        r['country']['value'],
            'country_code':   r['countryiso3code'],
            'indicator':      r['indicator']['value'],
            'indicator_code': r['indicator']['id'],
            'year':           int(r['date']),
            'value':          r['value']
        })

    df = pd.DataFrame(rows)
    df = df.sort_values(['country_code', 'year']).reset_index(drop=True)
    return df


# --- Cargar archivos disponibles (cuando existan) ---
wb_files = glob.glob(os.path.join(PATHS['worldbank'], '*.json'))

if wb_files:
    wb_frames = {os.path.basename(f): load_worldbank(os.path.basename(f)) for f in wb_files}
    for name, df in wb_frames.items():
        print(f"{name}: {df.shape}")
        display(df.head())
else:
    print("⚠️  Sin datos de World Bank aún. Ejecutar worldbank_api.py primero.")

NY_GDP_MKTP_CD_PL_2014_2023.json: (10, 6)


,country,country_code,indicator,indicator_code,year,value
0,Poland,POL,GDP (current US$),NY.GDP.MKTP.CD,2014,5.421342e+11
1,Poland,POL,GDP (current US$),NY.GDP.MKTP.CD,2015,4.800541e+11
2,Poland,POL,GDP (current US$),NY.GDP.MKTP.CD,2016,4.732596e+11
3,Poland,POL,GDP (current US$),NY.GDP.MKTP.CD,2017,5.283567e+11
4,Poland,POL,GDP (current US$),NY.GDP.MKTP.CD,2018,5.946167e+11


## 2 · Validación rápida

In [6]:
def validate(df, name):
    print(f"\n{'='*40}")
    print(f" {name}")
    print(f"{'='*40}")
    print(f"  Shape:   {df.shape}")
    print(f"  Dtypes:  {df.dtypes.to_dict()}")
    print(f"  Nulls:   {df.isnull().sum().to_dict()}")
    if 'date' in df.columns:
        print(f"  Rango:   {df['date'].min()} → {df['date'].max()}")
    elif 'year' in df.columns:
        print(f"  Rango:   {df['year'].min()} → {df['year'].max()}")

validate(nbp_eur,     'NBP — EUR/PLN')
validate(nbp_gold,    'NBP — Oro (PLN/g)')
validate(df_eurostat, 'Eurostat')


 NBP — EUR/PLN
  Shape:   (10, 4)
  Dtypes:  {'date': dtype('<M8[ns]'), 'currency': dtype('O'), 'table': dtype('O'), 'rate': dtype('float64')}
  Nulls:   {'date': 0, 'currency': 0, 'table': 0, 'rate': 0}
  Rango:   2026-06-10 00:00:00 → 2026-06-23 00:00:00

 NBP — Oro (PLN/g)
  Shape:   (5, 2)
  Dtypes:  {'date': dtype('<M8[ns]'), 'gold_pln': dtype('float64')}
  Nulls:   {'date': 0, 'gold_pln': 0}
  Rango:   2026-06-17 00:00:00 → 2026-06-23 00:00:00

 Eurostat
  Shape:   (717, 11)
  Dtypes:  {'time': dtype('O'), 'time_label': dtype('O'), 'geo': dtype('O'), 'geo_label': dtype('O'), 'coicop': dtype('O'), 'coicop_label': dtype('O'), 'unit': dtype('O'), 'unit_label': dtype('O'), 'freq': dtype('O'), 'freq_label': dtype('O'), 'value': dtype('float64')}
  Nulls:   {'time': 0, 'time_label': 0, 'geo': 0, 'geo_label': 0, 'coicop': 0, 'coicop_label': 0, 'unit': 0, 'unit_label': 0, 'freq': 0, 'freq_label': 0, 'value': 0}


## 3 · Análisis
> Agregar celdas por módulo temático: Economía, Demografía, Vivienda, Energía, Mercado laboral.